<img src="https://raw.githubusercontent.com/LNB-DH/BSSDH_2026_LLM_API_workshop/main/img/session_2/session2_start.png" alt="Intro to Session 2" width="100%">

# Using LLMs in Humanities Research via API

## Session 2 (14.00–15.30) — Prompt engineering for textual data

Through practical examples, we will explore prompt engineering techniques for concept mining and named entity recognition in textual data. We will move from one document to a small, auditable batch while keeping every generated claim connected to its historical source.

### Session outline

- **Prompt engineering:** turn a research idea into explicit instructions, inclusion and exclusion criteria, and a stable response format.
- **Named entity recognition (NER):** identify and classify people, organizations, places, and other named entities.
- **Concept mining:** move from open-ended themes to a clearly defined research concept.
- **Evaluation and batch processing:** make predictions, test one document, check evidence, revise the prompt, and then process a small sample.

This session also prepares you for the independent assignment used in Session 3. The worked examples here are deliberately different from the six assignment excerpts.


In [ ]:
# This is the notebook's only text-entry prompt.
import getpass

OPENROUTER_API_KEY = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()
if not OPENROUTER_API_KEY:
    raise ValueError('No API key was entered.')
print('API key received in runtime memory: PASS')


## 1. Reusing the 2025 workshop corpus

We will continue using the public [BSSDH 2025 workshop data repository](https://github.com/LNB-DH/BSSDH_2025_workshop_data) and its `Latvian_Economic_Review_1936_1940.zip` archive. Reusing the same fixed source makes comparisons with last year's exercises possible.

The *Latvian Economic Review* (LERQ) is an English-language quarterly published from 1936 to 1940. The archive contains 419 OCR-derived text segments with title, author, and National Library of Latvia URI metadata. The source wording has not been normalized, so line wrapping and occasional OCR errors remain part of the research material.

Filename example: `lerq1936s01n02_031_plaintext_s17.txt` is segment 17 from page 31 of issue 2, 1936.


In [ ]:
# Imports and configuration used throughout Session 2.
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import re
import time

import pandas as pd
import requests
from IPython.display import display
from openai import OpenAI

DATA_URL = (
    'https://github.com/LNB-DH/BSSDH_2025_workshop_data/'
    'raw/main/data/Latvian_Economic_Review_1936_1940.zip'
)
EXPECTED_ARCHIVE_SHA256 = 'e0b03c2505de52c57e30767569fabf7ddea5d53b4000ffef647ce5d0812a71e4'
EXPECTED_TEXT_FILE_COUNT = 419
DATA_DIR = Path('data')
LERQ_DIR = DATA_DIR / 'Latvian_Economic_Review'

MODEL_ID = 'google/gemini-3.5-flash-lite'
MODEL_LAST_VERIFIED = '2026-08-04'

print(f'Core model: {MODEL_ID} (verified {MODEL_LAST_VERIFIED})')
print(f'Corpus directory: {LERQ_DIR}')


In [ ]:
# Download only when the expected corpus is not already available.
existing_text_files = sorted(LERQ_DIR.rglob('*.txt')) if LERQ_DIR.exists() else []

if len(existing_text_files) == EXPECTED_TEXT_FILE_COUNT:
    print(f'Corpus already available: {len(existing_text_files)} text files.')
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Downloading the LERQ archive from {DATA_URL}')
    download = requests.get(DATA_URL, timeout=90)
    download.raise_for_status()
    archive_bytes = download.content
    archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
    if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
        raise ValueError(
            'The downloaded archive does not match the workshop checksum. '
            'Stop and verify whether the data repository has changed.'
        )
    with ZipFile(BytesIO(archive_bytes)) as archive:
        archive.extractall(DATA_DIR)
    print('Download checksum: PASS')

text_files = sorted(LERQ_DIR.rglob('*.txt'))
if len(text_files) != EXPECTED_TEXT_FILE_COUNT:
    raise ValueError(
        f'Expected {EXPECTED_TEXT_FILE_COUNT} LERQ text files, found {len(text_files)}.'
    )

print(f'Corpus verification: PASS ({len(text_files)} text files)')
print('First three files:')
for path in text_files[:3]:
    print(' -', path.name)


## 2. Two deliberately selected documents

We will not select an arbitrary 'sixth file.' Instead, we use a positive/negative teaching pair:

1. **State Land Bank 15 Years Activity** contains a person, an organization, places, and explicit discussion of rural credit and agrarian reform.
2. **Gypsum** contains several place names but should not count as evidence of rural credit or agriculture merely because it discusses exports.

Read both before prompting the model. The source text—not the model response—is our evidence.


In [ ]:
TRAINING_FILENAMES = {
    'state_land_bank': 'lerq1937s01n08_029_plaintext_s09.txt',
    'gypsum': 'lerq1936s01n01_014_plaintext_s06.txt',
}

def load_lerq_document(filename):
    path = LERQ_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing workshop document: {path}')
    return path.read_text(encoding='utf-8')

training_documents = {
    name: load_lerq_document(filename)
    for name, filename in TRAINING_FILENAMES.items()
}

for short_name, document_text in training_documents.items():
    print('=' * 78)
    print(f'{short_name}: {TRAINING_FILENAMES[short_name]}')
    print('=' * 78)
    print(document_text)


### Predict before calling the model

Before running any API cell, write brief predictions in your own notebook copy:

| Document | Which entity types do you expect? | Is rural credit present, absent, or unclear? | Why? |
|---|---|---|---|
| State Land Bank | [WRITE] | [WRITE] | [WRITE] |
| Gypsum | [WRITE] | [WRITE] | [WRITE] |

Do not rewrite a prediction after seeing the model output. A disagreement gives us something useful to investigate.


## 3. Configure the OpenRouter client

The workshop core model is [Google Gemini 3.5 Flash Lite](https://openrouter.ai/google/gemini-3.5-flash-lite), accessed through OpenRouter. The model name and verification date are kept in one configuration cell because model availability and pricing change.

You entered the key in the first code cell. It remains only in runtime memory: it is not displayed and is not written to `.env` or to the notebook. Every remaining code cell runs without typed confirmation.


In [ ]:
# Prepared API helpers. Read them now; you do not need to rewrite them.
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
    default_headers={
        'HTTP-Referer': 'https://digitalhumanities.lv/en/bssdh/2026/',
        'X-OpenRouter-Title': 'BSSDH 2026 LLM API Workshop',
    },
)

if 'api_records' not in globals():
    api_records = {}

def build_prompt_id(system_prompt, task):
    prompt_bytes = f'{system_prompt}\n\n{task}'.encode('utf-8')
    return hashlib.sha256(prompt_bytes).hexdigest()[:12]

def make_request_record(source_name, source_text, system_prompt, task,
                        model=MODEL_ID, max_completion_tokens=700, temperature=0.1):
    prompt_id = build_prompt_id(system_prompt, task)
    messages = [
        {'role': 'system', 'content': system_prompt},
        {
            'role': 'user',
            'content': f'SOURCE ID: {source_name}\n\nSOURCE:\n{source_text}\n\nTASK:\n{task}',
        },
    ]
    started = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_completion_tokens=max_completion_tokens,
            temperature=temperature,
        )
        raw_text = response.choices[0].message.content or ''
        if not raw_text.strip():
            raise RuntimeError('The API returned an empty response.')
        usage = response.usage.model_dump(mode='json') if response.usage else {}
        return {
            'source_name': source_name,
            'prompt_id': prompt_id,
            'request_status': 'received',
            'request_error': '',
            'raw_response': raw_text,
            'requested_model': model,
            'returned_model': response.model,
            'response_id': response.id,
            'usage': usage,
            'elapsed_seconds': round(time.perf_counter() - started, 2),
        }
    except Exception as exc:
        return {
            'source_name': source_name,
            'prompt_id': prompt_id,
            'request_status': 'failed',
            'request_error': f'{type(exc).__name__}: {exc}',
            'raw_response': '',
            'requested_model': model,
            'returned_model': '',
            'response_id': '',
            'usage': {},
            'elapsed_seconds': round(time.perf_counter() - started, 2),
        }

def run_one_cached(record_key, source_name, source_text, system_prompt, task):
    prompt_id = build_prompt_id(system_prompt, task)
    existing = api_records.get(record_key)
    same_successful_prompt = (
        existing is not None
        and existing.get('prompt_id') == prompt_id
        and existing.get('request_status') == 'received'
    )
    if same_successful_prompt:
        print(f'Reusing the successful response for {source_name} (prompt ID: {prompt_id}).')
        return existing

    print(f'Calling {MODEL_ID} for {source_name} (prompt ID: {prompt_id})...')
    record = make_request_record(source_name, source_text, system_prompt, task)
    api_records[record_key] = record
    print(f"Request status: {record['request_status']}")
    if record['request_status'] == 'failed':
        print(record['request_error'])
    return record

def show_record(record):
    if not record:
        print('No current response to display.')
    elif record['request_status'] == 'failed':
        print('Request failed:', record['request_error'])
    else:
        print(f"Prompt ID: {record['prompt_id']}")
        print(f"Returned model: {record['returned_model']}")
        print(f"Elapsed: {record['elapsed_seconds']} seconds")
        print(record['raw_response'])

print('API helpers ready. No model request has been sent yet.')


## 4. Named entity recognition: start with a minimal prompt

Named entities are specific named people, organizations, places, works, products, events, and similar referents. NER differs from concept mining: `State Land Bank` is an organization mention, while `rural credit` is an abstract concept.

We first use an intentionally minimal prompt. Its weaknesses give us evidence for the next prompt revision.


In [ ]:
NER_BASELINE_SYSTEM = (
    'You are a digital humanities research assistant. Extract the named entities '
    'from the supplied historical text.'
)
NER_TASK = 'Identify the named entities in the source.'

ner_baseline_record = run_one_cached(
    record_key='ner_baseline_state_land_bank',
    source_name=TRAINING_FILENAMES['state_land_bank'],
    source_text=training_documents['state_land_bank'],
    system_prompt=NER_BASELINE_SYSTEM,
    task=NER_TASK,
)
show_record(ner_baseline_record)


### Evaluate the baseline response

Compare it with the source rather than asking whether it merely looks plausible:

- Did it preserve the exact spelling used in the document?
- Did it distinguish a person, organization, and place?
- Did it add outside information?
- Did it duplicate an entity or confuse an abstract concept with an entity?
- Is the format consistent enough to parse automatically?

A better prompt needs a task definition, allowed labels, evidence rules, an abstention rule, and an output contract.


### Refine one variable: the prompt

We keep the document, model, and sampling parameters fixed. Only the prompt changes, so differences in the response are easier to interpret.


In [ ]:
NER_REFINED_SYSTEM = '''You are a digital humanities researcher performing named entity recognition.
Use only the supplied source. Do not add facts from outside knowledge.

Extract explicit named mentions and assign exactly one category:
- PERSON: a named human being
- ORGANIZATION: a named institution, company, association, or government body
- PLACE: a named country, region, city, or other geographic place
- MISC: another proper name that does not fit the categories above

Preserve the spelling in the source. Do not treat generic roles, unnamed groups, dates,
currencies, or abstract concepts as named entities. Deduplicate repeated mentions.
For evidence, copy a short passage containing the mention.
Return only a valid JSON array. Each item must contain exactly these keys:
mention, category, evidence. Return [] if no named entity is present.'''

ner_refined_record = run_one_cached(
    record_key='ner_refined_state_land_bank',
    source_name=TRAINING_FILENAMES['state_land_bank'],
    source_text=training_documents['state_land_bank'],
    system_prompt=NER_REFINED_SYSTEM,
    task=NER_TASK,
)
show_record(ner_refined_record)


In [ ]:
# Parse the structure and check whether each quoted passage occurs in the source.
def parse_json_value(raw_text):
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    object_start = cleaned.find('{')
    array_start = cleaned.find('[')
    starts = [position for position in (object_start, array_start) if position >= 0]
    if not starts:
        raise ValueError('No JSON object or array found.')
    start = min(starts)
    closing = '}' if cleaned[start] == '{' else ']'
    end = cleaned.rfind(closing)
    if end < start:
        raise ValueError('The JSON value is incomplete.')
    return json.loads(cleaned[start:end + 1])

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

def evidence_found_in_source(evidence, source_text):
    evidence_normalized = normalize_whitespace(evidence)
    source_normalized = normalize_whitespace(source_text)
    return bool(evidence_normalized) and evidence_normalized in source_normalized

if ner_refined_record and ner_refined_record['request_status'] == 'received':
    try:
        ner_items = parse_json_value(ner_refined_record['raw_response'])
        if not isinstance(ner_items, list):
            raise ValueError('The NER response is not a JSON array.')
        allowed_categories = {'PERSON', 'ORGANIZATION', 'PLACE', 'MISC'}
        ner_rows = []
        for item in ner_items:
            if not isinstance(item, dict):
                raise ValueError('Every NER item must be a JSON object.')
            if set(item) != {'mention', 'category', 'evidence'}:
                raise ValueError(f'Unexpected NER keys: {sorted(item)}')
            category = str(item['category']).strip().upper()
            if category not in allowed_categories:
                raise ValueError(f'Unexpected category: {category}')
            ner_rows.append({
                'mention': str(item['mention']).strip(),
                'category': category,
                'evidence': str(item['evidence']).strip(),
                'evidence_found_in_source': evidence_found_in_source(
                    item['evidence'], training_documents['state_land_bank']
                ),
            })
        ner_table = pd.DataFrame(ner_rows)
        display(ner_table)
    except Exception as exc:
        print(f'NER validation failed: {type(exc).__name__}: {exc}')
        print('Inspect the raw response above before changing any parser.')
else:
    print('Run the refined NER call before validating it.')


## 5. Concept mining: from discovery to a research decision

Concept mining seeks abstract ideas, processes, and themes rather than only proper names. A text can discuss `agrarian reconstruction`, `rural indebtedness`, or `state-supported credit` without using our preferred modern research label.

We begin inductively: ask the model what concepts appear, while still requiring evidence. The result is exploratory and should not yet be treated as a stable classification.


In [ ]:
CONCEPT_DISCOVERY_SYSTEM = '''You are a digital humanities researcher exploring a historical economic text.
Identify up to six important economic or social concepts supported by the supplied source.
For each concept, give a short name, a short exact evidence passage, and one sentence
explaining the connection. Do not add concepts based only on outside knowledge.'''
CONCEPT_DISCOVERY_TASK = 'What important economic or social concepts are supported by this source?'

concept_discovery_record = run_one_cached(
    record_key='concept_discovery_state_land_bank',
    source_name=TRAINING_FILENAMES['state_land_bank'],
    source_text=training_documents['state_land_bank'],
    system_prompt=CONCEPT_DISCOVERY_SYSTEM,
    task=CONCEPT_DISCOVERY_TASK,
)
show_record(concept_discovery_record)


### From open-ended discovery to predetermined categories

The LERQ corpus could support many research domains: finance and banking, trade, agriculture and rural economy, industry, labour, prices, governance, international relations, education, and urban development.

Open-ended discovery helps generate hypotheses, but a corpus-level study usually needs an operational definition: explicit rules that another researcher could inspect and apply. For the worked example, we narrow the broad agriculture domain to **rural credit**.


## 6. Operationalize the concept: rural credit

**Research question:** Does the document describe credit, loans, or debt arrangements connected to agriculture or the rural economy?

Count **rural credit as present** when the source directly connects a financial mechanism—such as loans, mortgages, debt conversion, interest support, or agricultural banking—to farms, farmers, agricultural production, land acquisition, or rural industries.

Count it as **absent** when the source discusses:

- agriculture without a credit, loan, or debt mechanism;
- banking, trade, prices, or exports without a direct rural or agricultural connection;
- a metaphorical or inferred connection not supported by the wording.

Use **unclear** when damaged, incomplete, or ambiguous wording prevents a defensible decision. `Unclear` is not a substitute for `absent`.


### Record your classification predictions now

Complete this in your notebook copy before the next API cell.

| Document | Predicted label | Source-based reason |
|---|---|---|
| State Land Bank | [present / absent / unclear] | [WRITE] |
| Gypsum | [present / absent / unclear] | [WRITE] |


## 7. Test one document before running a batch

The operational definition becomes the analytical part of the system prompt. A separate output contract controls the response shape. Keeping these roles visible makes the prompt easier to revise.

The final four fields match the assignment used in Session 3.


In [ ]:
RURAL_CREDIT_DEFINITION = '''You are a careful digital humanities researcher.
Use only the supplied historical source. Decide whether the concept RURAL CREDIT is present.

Count it as present only when the source directly connects credit, loans, mortgages,
debt conversion, interest support, or agricultural banking to farms, farmers,
agricultural production, land acquisition, or rural industries.

Count it as absent when agriculture appears without a financial mechanism, or when
banking, trade, prices, and exports lack a direct rural or agricultural connection.
Do not infer a connection from outside knowledge.
Use unclear only when damaged, incomplete, or ambiguous wording prevents a decision.'''

CLASSIFICATION_CONTRACT = '''Return only one valid JSON object with exactly these fields:
{
  "label": "present, absent, or unclear",
  "confidence": 0,
  "evidence": "a short exact quotation from the source, or an empty string",
  "reason": "one concise sentence explaining the decision"
}
The label must be exactly present, absent, or unclear.
Confidence must be a whole number from 0 to 100.
Do not wrap the JSON in Markdown fences or add other text.'''

RURAL_CREDIT_SYSTEM = RURAL_CREDIT_DEFINITION + '\n\n' + CLASSIFICATION_CONTRACT
CLASSIFICATION_TASK = 'Classify this document for the concept rural credit.'

def classification_key(filename):
    return f'rural_credit::{filename}'

state_credit_record = run_one_cached(
    record_key=classification_key(TRAINING_FILENAMES['state_land_bank']),
    source_name=TRAINING_FILENAMES['state_land_bank'],
    source_text=training_documents['state_land_bank'],
    system_prompt=RURAL_CREDIT_SYSTEM,
    task=CLASSIFICATION_TASK,
)
show_record(state_credit_record)


In [ ]:
ALLOWED_LABELS = {'present', 'absent', 'unclear'}

def parse_classification_record(record, source_text):
    base = {
        'label': '',
        'confidence': None,
        'evidence': '',
        'reason': '',
        'parse_status': 'not attempted',
        'parse_error': '',
        'evidence_found_in_source': None,
    }
    if not record:
        return {**base, 'request_status': 'missing'}
    if record.get('request_status') != 'received':
        return {
            **base,
            'request_status': 'failed',
            'parse_error': record.get('request_error', ''),
        }
    try:
        parsed = parse_json_value(record['raw_response'])
        if not isinstance(parsed, dict):
            raise ValueError('The classification response is not a JSON object.')
        expected_keys = {'label', 'confidence', 'evidence', 'reason'}
        if set(parsed) != expected_keys:
            raise ValueError(f'Unexpected classification keys: {sorted(parsed)}')
        label = str(parsed['label']).strip().lower()
        if label not in ALLOWED_LABELS:
            raise ValueError(f'Unexpected label: {label!r}')
        confidence = int(float(parsed['confidence']))
        if not 0 <= confidence <= 100:
            raise ValueError('Confidence is outside 0–100.')
        evidence = str(parsed['evidence']).strip()
        return {
            'label': label,
            'confidence': confidence,
            'evidence': evidence,
            'reason': str(parsed['reason']).strip(),
            'parse_status': 'parsed',
            'parse_error': '',
            'evidence_found_in_source': (
                evidence_found_in_source(evidence, source_text) if evidence else None
            ),
            'request_status': 'received',
        }
    except Exception as exc:
        return {
            **base,
            'request_status': 'received',
            'parse_status': 'failed',
            'parse_error': f'{type(exc).__name__}: {exc}',
        }

state_credit_parsed = parse_classification_record(
    state_credit_record, training_documents['state_land_bank']
)
display(pd.DataFrame([state_credit_parsed]))
print('Model confidence is self-reported; it is not a measured probability of correctness.')


### Review the test before continuing

Compare the response with your prediction and the source:

- Is the label consistent with the operational definition?
- Is the evidence copied from the document?
- Does the reason explain the rule rather than add historical background?
- Would another researcher understand what counted and what did not?

If the prompt needs revision, change `RURAL_CREDIT_DEFINITION`, rerun the prompt cell, and retest this one document. The prompt ID prevents an older response from being silently reused with new instructions.


In [ ]:
# After accepting the prompt, test the deliberately negative Gypsum document.
gypsum_credit_record = run_one_cached(
    record_key=classification_key(TRAINING_FILENAMES['gypsum']),
    source_name=TRAINING_FILENAMES['gypsum'],
    source_text=training_documents['gypsum'],
    system_prompt=RURAL_CREDIT_SYSTEM,
    task=CLASSIFICATION_TASK,
)
show_record(gypsum_credit_record)

gypsum_credit_parsed = parse_classification_record(
    gypsum_credit_record, training_documents['gypsum']
)
display(pd.DataFrame([gypsum_credit_parsed]))


## 8. Run the accepted prompt across six different documents

A batch should follow—not replace—the one-document test. The six files below include likely positive and negative cases for rural credit. They are different from every source file in the Session 3 assignment.

The next cell reuses every successful response whose prompt ID still matches, then automatically processes only the remaining files. No additional text entry is required.


In [ ]:
BATCH_FILES = [
    'lerq1937s01n08_029_plaintext_s09.txt',  # State Land Bank
    'lerq1937s01n05_037_plaintext_s18.txt',  # Latvian Agrarian Credit
    'lerq1938s01n04_034_plaintext_s37.txt',  # Land Bank Promotes Agriculture
    'lerq1936s01n01_014_plaintext_s06.txt',  # Gypsum
    'lerq1936s01n01_013_plaintext_s05.txt',  # Cement industry
    'lerq1936s01n01_009_plaintext_s04.txt',  # Motor transport
]

ASSIGNMENT_SOURCE_FILES = {
    'lerq1936s01n01_003_plaintext_s01.txt',
    'lerq1937s01n07_004_plaintext_s02.txt',
    'lerq1938s01n04_013_plaintext_s09.txt',
    'lerq1938s01n04_032_plaintext_s36.txt',
    'lerq1939s01n03_030_plaintext_s11.txt',
    'lerq1940s01n01_009_plaintext_s04.txt',
}

overlap = set(BATCH_FILES) & ASSIGNMENT_SOURCE_FILES
if overlap:
    raise ValueError(f'Session 2 must not use assignment sources: {sorted(overlap)}')

batch_documents = {filename: load_lerq_document(filename) for filename in BATCH_FILES}
batch_prompt_id = build_prompt_id(RURAL_CREDIT_SYSTEM, CLASSIFICATION_TASK)
pending_files = []
for filename in BATCH_FILES:
    existing = api_records.get(classification_key(filename))
    is_current = (
        existing is not None
        and existing.get('prompt_id') == batch_prompt_id
        and existing.get('request_status') == 'received'
    )
    if not is_current:
        pending_files.append(filename)

if not pending_files:
    print('All six successful responses already exist for the current prompt.')
else:
    print(f'Processing {len(pending_files)} remaining API call(s):')
    for filename in pending_files:
        print(' -', filename)
    for position, filename in enumerate(pending_files, start=1):
        print(f'[{position}/{len(pending_files)}] {filename}')
        record = make_request_record(
            source_name=filename,
            source_text=batch_documents[filename],
            system_prompt=RURAL_CREDIT_SYSTEM,
            task=CLASSIFICATION_TASK,
        )
        api_records[classification_key(filename)] = record
        print('   ', record['request_status'])
        if record['request_status'] == 'failed':
            print('   ', record['request_error'])
        time.sleep(0.5)
    print('Batch finished.')


In [ ]:
# Convert request records into an auditable table without hiding failures.
result_rows = []
for filename in BATCH_FILES:
    document_text = batch_documents[filename]
    record = api_records.get(classification_key(filename))
    parsed = parse_classification_record(record, document_text)
    title_line = document_text.splitlines()[0].removeprefix('title:').strip()
    result_rows.append({
        'source_filename': filename,
        'title': title_line,
        **parsed,
        'prompt_id': record.get('prompt_id') if record else '',
        'returned_model': record.get('returned_model') if record else '',
        'total_tokens': (record.get('usage') or {}).get('total_tokens') if record else None,
    })

batch_results = pd.DataFrame(result_rows)
display_columns = [
    'source_filename', 'title', 'label', 'confidence', 'evidence',
    'evidence_found_in_source', 'reason', 'request_status', 'parse_status'
]
display(batch_results[display_columns])

failed = batch_results[
    (batch_results['request_status'] != 'received')
    | (batch_results['parse_status'] != 'parsed')
]
if not failed.empty:
    print('Inspect these request or parsing failures rather than repairing them with another LLM:')
    display(failed[['source_filename', 'request_status', 'parse_status', 'parse_error']])


In [ ]:
# A compact summary prepares us to read the assignment's table and charts.
label_order = ['present', 'absent', 'unclear']
label_counts = (
    batch_results.loc[batch_results['parse_status'] == 'parsed', 'label']
    .value_counts()
    .reindex(label_order, fill_value=0)
)
display(label_counts.rename('documents').to_frame())

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print('Matplotlib is unavailable locally; the results table above is complete.')
else:
    ax = label_counts.plot(
        kind='bar',
        color=['#2a9d8f', '#e76f51', '#e9c46a'],
        figsize=(7, 4),
        title='Rural-credit labels in the Session 2 training sample',
    )
    ax.set_xlabel('Model label')
    ax.set_ylabel('Number of documents')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()


## 9. Debrief: what counts as success?

A parsed response is not automatically a correct research result. Review the table and discuss:

- Which labels agree with your reading of the documents?
- Is every non-empty evidence passage found in its source?
- Did the prompt distinguish rural credit from generic banking and generic agriculture?
- Did `unclear` mean genuine ambiguity rather than model hesitation?
- What can six selected documents not tell us about the full 419-document corpus?
- What should be manually validated before scaling up?

Prompt wording, model choice, historical OCR, document selection, and the operational definition all remain part of the method and its limitations.


## 10. Session 3 assignment handoff

In Session 3 you will work through [`assignment_llm_api.ipynb`](assignment_llm_api.ipynb). It uses six different curated LERQ excerpts. You will repeat the workflow practised here while making the analytical decisions yourself:

1. formulate a research question;
2. write an operational definition with inclusion and exclusion criteria;
3. make predictions before calling the model;
4. write and test your own prompt on one document;
5. process the six-document mini-corpus;
6. check evidence and interpret a table and charts;
7. explain limitations and submit one executed notebook.

The OCR, translation, and image-input notebook remains available as bonus material and will be refined separately.


## Optional extensions

After the core session, you could:

- compare the baseline and refined NER outputs with a small hand-annotated answer key;
- add API-level [structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs) after checking that the selected model supports them;
- export validated records as JSON Lines with source and prompt metadata;
- test a larger reproducible sample only after estimating calls, tokens, and manual-validation effort;
- revisit the Rigasche Zeitung bonus notebook for historical OCR, translation, and image-based analysis.

Do not run the full corpus merely because the code can. Sampling and validation are research-design decisions, not only technical parameters.
